# 01 · TCIA inventory

**Project:** Acquisition shortcuts in public cancer imaging: an archive-scale, metadata-only audit of The Cancer Imaging Archive (TCIA)
**Author:** Hamza El Massaoudy · **Notebook written:** 28 Oct 2026 · **Plan:** `00_RESEARCH_PLAN.md` (sections 6, 7, 7a)

**What this notebook does, in plain words**
1. It lists every public TCIA collection: how many patients, which scan types (CT, MR, ...), which body parts, and whether the subjects are humans, animals or phantoms.
2. For the candidate collections (all TCGA and CPTAC collections, plus the six others in plan section 6), it downloads the **scan descriptions only** (the "stamp" of each series: machine, software, number of images, ...). **No images are downloaded.**
3. It checks how complete those descriptions are, collection by collection.
4. It picks one **main series** per patient with the fixed rule from plan 7a.7.
5. For TCGA and CPTAC patients, it fetches the labels (grade, stage, histology) from the GDC API and counts how many patients each candidate task has per class (rule 7a.1: at least 20 per class).
6. It checks whether the study dates look real or shifted by de-identification (rule 7a.5), and whether any field could leak the diagnosis (rule 7a.3).
7. It records which planned collections are missing from TCIA's public API, and checks whether the Imaging Data Commons (IDC) has them.

**Outputs** go to Drive: `Acquisition-Shortcut-Audit/results/01_inventory/`. Every raw API reply is cached in `Acquisition-Shortcut-Audit/cache/api/`, so every number can be traced back, and a rerun takes about a minute.

**How to run:** in Colab, *Runtime → Run all*. No GPU is needed. Sign in with **the Google account that holds the project Drive folder** when Drive asks.

**Nothing here fits a model or looks at labels against metadata.** This notebook only takes stock. Modelling starts in notebook 03, after the rules in plan 7a are final.

In [1]:
# 1. Setup: Drive and folders
import os, re, sys, json, time, gzip, hashlib, datetime, platform
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
else:  # local test runs only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')

OUT = PROJECT / 'results' / '01_inventory'
CACHE = PROJECT / 'cache' / 'api'
LABELS = PROJECT / 'labels'
for p in (OUT, CACHE, LABELS):
    p.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
TCIA = 'https://services.cancerimagingarchive.net/nbia-api/services/v1'
GDC = 'https://api.gdc.cancer.gov'
print('Project folder:', PROJECT)
print('Outputs to    :', OUT)
print('Run started   :', RUN_STARTED, '(UTC)')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Acquisition-Shortcut-Audit
Outputs to    : /content/drive/MyDrive/Acquisition-Shortcut-Audit/results/01_inventory
Run started   : 2026-10-28T12:20:42+00:00 (UTC)


In [2]:
# 2. Packages
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

## Frozen settings
These come from plan sections 6 and 7a. Change them only with a dated entry in `00_LOG.md`.

In [3]:
# 3. Frozen settings (plan sections 6 and 7a)

# Non-TCGA candidates from plan section 6 (exact TCIA names) -> label planned
CANDIDATES_NON_TCGA = {
    'LIDC-IDRI': 'nodule malignancy rating (radiologists)',
    'Duke-Breast-Cancer-MRI': 'molecular subtype',
    'NSCLC Radiogenomics': 'histology; mutations',
    'PROSTATEx': 'clinical significance',
    'UPENN-GBM': 'molecular markers',
    'HCC-TACE-Seg': 'to define from the clinical file',
}
# Every TCGA-* and CPTAC-* collection is a candidate too: their labels come from the GDC API.

# Tasks where the class is the source collection (plan section 6b)
COLLECTION_DEFINED_TASKS = {
    'glioma_grade': ['TCGA-GBM', 'TCGA-LGG'],
    'renal_subtype': ['TCGA-KIRC', 'TCGA-KIRP', 'TCGA-KICH'],
    'lung_histology': ['TCGA-LUAD', 'TCGA-LUSC'],
}

MIN_PER_CLASS = 20             # rule 7a.1 (may be reset once, after this notebook, before notebook 03)
IMAGING_MODALITIES = ('CT', 'MR')
MIN_IMAGES = 10                # rule 7a.7: a main series has at least this many images

# Rule 7a.7: series that can never be the main series. Matched on SeriesDescription
# (on ProtocolName only when SeriesDescription is empty), case-insensitive.
EXCLUDE_SERIES_PATTERN = (r'scout|locali[sz]er|\bloc\b|topogram|survey|calibration|\bcal\b|'
                          r'bolus|smart\s*prep|monitor|dose\s*(?:report|info)|screen\s*save|'
                          r'secondary|derived|\bmpr\b|\bmip\b|reformat|fusion')
# For CT only: sagittal and coronal series are reformats of the axial scan (for MR they are real acquisitions)
CT_REFORMAT_PATTERN = r'\bsag|\bcor\b|coronal'
# Series that TCIA marks as third-party analysis results (ThirdPartyAnalysis = yes) are never the main series.

# For local test runs only: limit the candidate list (comma-separated collection names)
TEST_COLLECTIONS = [c for c in os.environ.get('ASA_TEST_COLLECTIONS', '').split(',') if c]

In [4]:
# 4. Robust, cached API access
SESSION = requests.Session()
SESSION.mount('https://', HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1.5, status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({'GET', 'POST'}))))
CALLS = {'network': 0, 'cache': 0}
FAILURES = []

def fetch_json(url, params=None, payload=None, timeout=300):
    """GET (or POST when payload is given) with retries. The raw reply is cached on Drive,
    so reruns are fast and every number can be traced to the exact API reply."""
    key = json.dumps({'url': url, 'params': params, 'payload': payload}, sort_keys=True)
    path = CACHE / (hashlib.sha1(key.encode()).hexdigest() + '.json.gz')
    if path.exists():
        CALLS['cache'] += 1
        with gzip.open(path, 'rt', encoding='utf-8') as f:
            return json.load(f)['data']
    if payload is None:
        r = SESSION.get(url, params=params, timeout=timeout)
    else:
        r = SESSION.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    text = r.text.strip()
    data = json.loads(text) if text else []   # TCIA answers an unknown collection with an empty body
    tmp = path.with_suffix('.tmp')
    with gzip.open(tmp, 'wt', encoding='utf-8') as f:
        json.dump({'request': json.loads(key),
                   'fetched_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
                   'data': data}, f)
    tmp.replace(path)
    CALLS['network'] += 1
    time.sleep(0.2)   # be polite to the public API
    return data

def tcia(endpoint, **params):
    return fetch_json(f'{TCIA}/{endpoint}', params=params or None)

def blank_to_nan(df):
    return df.replace(r'^\s*$', np.nan, regex=True)

## Step A · All public TCIA collections

In [5]:
# 5. Step A: every public collection, with patients, modalities and body parts
collections = sorted(c['Collection'] for c in tcia('getCollectionValues') if c.get('Collection'))
print(len(collections), 'public collections on the TCIA API')

rows = []
for i, col in enumerate(collections, 1):
    try:
        patients = tcia('getPatient', Collection=col)
        mods = sorted({m.get('Modality') for m in tcia('getModalityValues', Collection=col)} - {None, ''})
        parts = sorted({b.get('BodyPartExamined') for b in tcia('getBodyPartValues', Collection=col)} - {None, ''})
    except Exception as e:
        FAILURES.append({'step': 'A', 'collection': col, 'error': repr(e)})
        continue
    species = sorted({p.get('SpeciesDescription') for p in patients} - {None, ''})
    phantom = any(str(p.get('Phantom', '')).upper() == 'YES' for p in patients)
    if phantom:
        subject = 'phantom'
    elif species == ['Homo sapiens']:
        subject = 'human'
    elif species:
        subject = 'animal or mixed'
    else:
        subject = 'unknown'
    rows.append({'collection': col,
                 'n_patients': len({p.get('PatientId') for p in patients}),
                 'modalities': ';'.join(mods),
                 'has_CT': 'CT' in mods, 'has_MR': 'MR' in mods,
                 'body_parts': ';'.join(parts),
                 'subject_type': subject, 'species': ';'.join(species),
                 'is_tcga': col.startswith('TCGA-'), 'is_cptac': col.startswith('CPTAC-')})
    if i % 25 == 0:
        print(f'  {i}/{len(collections)} collections done')

inv = pd.DataFrame(rows)
inv['human_ct_or_mr'] = (inv.has_CT | inv.has_MR) & (inv.subject_type == 'human')
inv.to_csv(OUT / 'collections_inventory.csv', index=False)

print('\nSubject types:', inv.subject_type.value_counts().to_dict())
print('Human collections with CT or MR:', int(inv.human_ct_or_mr.sum()))
show(inv[inv.human_ct_or_mr].sort_values('n_patients', ascending=False))

156 public collections on the TCIA API
  25/156 collections done
  50/156 collections done
  75/156 collections done
  100/156 collections done
  125/156 collections done
  150/156 collections done

Subject types: {'human': 132, 'phantom': 14, 'animal or mixed': 10}
Human collections with CT or MR: 114


,collection,n_patients,modalities,has_CT,has_MR,body_parts,subject_type,species,is_tcga,is_cptac,human_ct_or_mr
59,EAY131,2813,CT;MR;NM;PT;RTSTRUCT;SEG;XA,True,True,ABDOMEN;ABDOMENPELVIS;ANKLE;AORTA;BONYPELVIS;BRAIN;CHEST...,human,Homo sapiens,False,False,True
39,COVID-19-NY-SBU,1384,CR;CT;DX;MR;NM;OT;PT;SR,True,True,ABDOMEN;ABDOMENPELVIS;CHEST;CHESTABDPELVIS;KIDNEY,human,Homo sapiens,False,False,True
109,Prostate-MRI-US-Biopsy,1151,MR;US,False,True,ABDOMEN;PELVIS;PROSTATE;TSPINE,human,Homo sapiens,False,False,True
68,LIDC-IDRI,1010,CR;CT;DX;SEG;SR,True,False,CHEST,human,Homo sapiens,False,False,True
4,ACRIN-Contralateral-Breast-MR,984,CR;MR,False,True,BREAST,human,Homo sapiens,False,False,True
57,Duke-Breast-Cancer-MRI,922,MR;SEG,False,True,BREAST,human,Homo sapiens,False,False,True
60,FDG-PET-CT-Lesions,900,CT;PT;SEG,True,False,ABDOMEN;WHOLEBODY,human,Homo sapiens,False,False,True
49,CT COLONOGRAPHY,825,CT,True,False,COLON,human,Homo sapiens,False,False,True
65,ISPY2,719,MR;SEG,False,True,BREAST;TSPINE,human,Homo sapiens,False,False,True
15,Advanced-MRI-Breast-Lesions,632,MR;SEG,False,True,BREAST,human,Homo sapiens,False,False,True


In [6]:
# 6. Planned collections that are missing from the public TCIA API
planned = set(CANDIDATES_NON_TCGA) | {c for cols in COLLECTION_DEFINED_TASKS.values() for c in cols}
missing = pd.DataFrame([{'collection': c,
                         'planned_for': '; '.join([t for t, cols in COLLECTION_DEFINED_TASKS.items() if c in cols]
                                                  + (['label task (plan 6a)'] if c in CANDIDATES_NON_TCGA else []))}
                        for c in sorted(planned - set(collections))], columns=['collection', 'planned_for'])
missing.to_csv(OUT / 'candidates_missing.csv', index=False)
print('Planned collections NOT on the public TCIA API:', missing.collection.tolist() or 'none')

Planned collections NOT on the public TCIA API: ['TCGA-GBM', 'TCGA-LGG']


## Step B · Series and study metadata for the candidate collections
Only text descriptions are downloaded, never images.

In [7]:
# 7. Step B: series-level and study-level metadata
candidates = [c for c in collections if c.startswith(('TCGA-', 'CPTAC-'))] + \
             [c for c in CANDIDATES_NON_TCGA if c in collections]
if TEST_COLLECTIONS:
    candidates = [c for c in candidates if c in TEST_COLLECTIONS]
    print('TEST RUN: candidates limited to', candidates)
print(len(candidates), 'candidate collections')

series_parts, study_parts = [], []
for i, col in enumerate(candidates, 1):
    try:
        s = pd.DataFrame(tcia('getSeries', Collection=col))
        st = pd.DataFrame(tcia('getPatientStudy', Collection=col))
    except Exception as e:
        FAILURES.append({'step': 'B', 'collection': col, 'error': repr(e)})
        continue
    s['Collection'], st['Collection'] = col, col
    series_parts.append(s)
    study_parts.append(st)
    print(f'  {i:>2}/{len(candidates)} {col:<26} {len(s):>6} series  {len(st):>5} studies')

series = blank_to_nan(pd.concat(series_parts, ignore_index=True))
studies = blank_to_nan(pd.concat(study_parts, ignore_index=True))
series['ImageCount'] = pd.to_numeric(series['ImageCount'], errors='coerce')
series['StudyDate_dt'] = pd.to_datetime(series['StudyDate'], errors='coerce')
series.drop(columns='StudyDate_dt').to_csv(OUT / 'series_metadata.csv.gz', index=False)
studies.to_csv(OUT / 'studies.csv.gz', index=False)
print(f'\nSaved {len(series):,} series and {len(studies):,} studies '
      f'from {series.PatientID.nunique():,} patients.')

33 candidate collections
   1/33 CPTAC-AML                      47 series      9 studies
   2/33 CPTAC-CCRCC                  1502 series     97 studies
   3/33 CPTAC-CM                      264 series     24 studies
   4/33 CPTAC-LSCC                    233 series     43 studies
   5/33 CPTAC-LUAD                    448 series     75 studies
   6/33 CPTAC-PDA                    1669 series    134 studies
   7/33 CPTAC-SAR                     265 series     33 studies
   8/33 CPTAC-STAD                     99 series     22 studies
   9/33 CPTAC-UCEC                   2576 series    146 studies
  10/33 TCGA-BLCA                    1051 series    192 studies
  11/33 TCGA-BRCA                    2041 series    164 studies
  12/33 TCGA-CESC                     488 series     57 studies
  13/33 TCGA-COAD                      93 series     32 studies
  14/33 TCGA-ESCA                     126 series     17 studies
  15/33 TCGA-KICH                     109 series     15 studies
  16/33 TCGA-KI

In [8]:
# 8. Role of every metadata field (rules 7a.3 and 7a.4). Unknown fields are flagged for review.
FIELD_ROLES = {
    'Collection': 'blocked: names the source (7a.3)',
    'CollectionURI': 'blocked: names the source (7a.3)',
    'DateReleased': 'blocked: release date (7a.3)',
    'TimeStamp': 'blocked: TCIA ingest time (7a.3)',
    'LicenseName': 'blocked: license (7a.3)',
    'LicenseURI': 'blocked: license (7a.3)',
    'PatientID': 'identifier: blocked (TCGA site code used only for grouping)',
    'PatientName': 'identifier: blocked',
    'StudyInstanceUID': 'identifier: blocked',
    'SeriesInstanceUID': 'identifier: blocked',
    'AdmittingDiagnosesDescription': 'blocked: may name the diagnosis (7a.3)',
    'Modality': 'scanner/protocol',
    'Manufacturer': 'scanner/protocol',
    'ManufacturerModelName': 'scanner/protocol',
    'SoftwareVersions': 'scanner/protocol',
    'BodyPartExamined': 'scanner/protocol',
    'SeriesNumber': 'scanner/protocol',
    'ImageCount': 'scanner/protocol (main series)',
    'SeriesDescription': 'scanner/protocol: text, label-word blocklist',
    'ProtocolName': 'scanner/protocol: text, label-word blocklist',
    'StudyDesc': 'scanner/protocol: text, label-word blocklist',
    'StudyDescription': 'scanner/protocol: text, label-word blocklist',
    'SeriesCount': 'care pathway (7a.4)',
    'LongitudinalTemporalEventType': 'care pathway: reference event for the offset (7a.4)',
    'LongitudinalTemporalOffsetFromEvent': 'care pathway: days from the event, e.g. diagnosis (7a.4)',
    'ThirdPartyAnalysis': 'blocked: TCIA curation flag; flagged series are never the main series',
    'StudyDate': 'date: only if judged real (7a.5)',
    'SeriesDate': 'date: only if judged real (7a.5)',
    'FileSize': 'sensitivity analysis only (7a.3)',
    'PatientAge': 'demographic: not an acquisition feature',
    'PatientSex': 'demographic: not an acquisition feature',
    'EthnicGroup': 'demographic: not an acquisition feature',
}
roles = []
for table, df in (('series', series.drop(columns='StudyDate_dt')), ('studies', studies)):
    for c in df.columns:
        roles.append({'table': table, 'field': c,
                      'role': FIELD_ROLES.get(c, 'UNCLASSIFIED: review before notebook 03'),
                      'fraction_filled': round(float(df[c].notna().mean()), 4)})
roles = pd.DataFrame(roles)
roles.to_csv(OUT / 'field_roles.csv', index=False)
unclassified = roles[roles.role.str.startswith('UNCLASSIFIED')]
print('Fields returned by the API:', len(roles), '| unclassified:', unclassified.field.tolist() or 'none')

Fields returned by the API: 38 | unclassified: ['AnnotationsFlag', 'StudyID']


## Step C · How complete is the metadata?

In [9]:
# 9. Step C: metadata coverage per collection (CT and MR series only)
img = series[series.Modality.isin(IMAGING_MODALITIES)].copy()
# Vendor normalised to its first word, for this inventory only (e.g. 'Philips Healthcare' -> 'PHILIPS')
img['vendor'] = img.Manufacturer.str.upper().str.extract(r'^\s*([A-Z]+)', expand=False)
COVERAGE_FIELDS = ['Manufacturer', 'ManufacturerModelName', 'SoftwareVersions',
                   'SeriesDescription', 'ProtocolName', 'BodyPartExamined', 'ImageCount']
cov = []
for col in candidates:
    a = series[series.Collection == col]
    g = img[img.Collection == col]
    row = {'collection': col,
           'n_patients_any': a.PatientID.nunique(),
           'n_patients_CT': g[g.Modality == 'CT'].PatientID.nunique(),
           'n_patients_MR': g[g.Modality == 'MR'].PatientID.nunique(),
           'n_series_CT_MR': len(g), 'n_studies_CT_MR': g.StudyInstanceUID.nunique(),
           'n_vendors': g.vendor.nunique(), 'n_manufacturer_strings': g.Manufacturer.nunique(),
           'n_models': g.ManufacturerModelName.nunique(), 'n_software_versions': g.SoftwareVersions.nunique()}
    for f in COVERAGE_FIELDS:
        row[f'filled_{f}'] = round(float(g[f].notna().mean()), 3) if len(g) else np.nan
    cov.append(row)
cov = pd.DataFrame(cov)
cov.to_csv(OUT / 'metadata_coverage.csv', index=False)
show(cov, 40)

,collection,n_patients_any,n_patients_CT,n_patients_MR,n_series_CT_MR,n_studies_CT_MR,n_vendors,n_manufacturer_strings,n_models,n_software_versions,filled_Manufacturer,filled_ManufacturerModelName,filled_SoftwareVersions,filled_SeriesDescription,filled_ProtocolName,filled_BodyPartExamined,filled_ImageCount
0,CPTAC-AML,5,5,0,45,7,2,2,2,3,1.000,1.000,0.867,1.000,1.000,1.000,1.0
1,CPTAC-CCRCC,71,64,14,866,97,5,6,42,40,1.000,1.000,0.904,0.963,1.000,0.590,1.0
2,CPTAC-CM,18,17,1,248,24,4,5,10,13,0.827,1.000,1.000,0.972,1.000,0.706,1.0
3,CPTAC-LSCC,36,36,0,213,43,5,5,19,25,0.643,1.000,0.958,0.770,1.000,0.643,1.0
4,CPTAC-LUAD,38,37,2,425,75,4,6,36,37,0.972,1.000,0.875,0.951,1.000,0.711,1.0
5,CPTAC-PDA,110,99,17,1122,131,4,5,45,47,0.758,1.000,0.764,0.755,0.545,0.843,1.0
6,CPTAC-SAR,24,18,11,259,33,4,4,19,24,0.950,1.000,0.915,0.934,1.000,0.158,1.0
7,CPTAC-STAD,20,19,0,98,21,3,4,7,7,1.000,1.000,0.980,1.000,0.888,1.000,1.0
8,CPTAC-UCEC,90,75,36,1950,144,3,3,22,22,0.634,1.000,0.953,1.000,0.947,0.873,1.0
9,TCGA-BLCA,120,107,20,1004,167,6,8,39,49,0.999,0.999,0.948,0.947,0.716,0.839,1.0


## Step D · One main series per patient (rule 7a.7)
Applied separately for CT and for MR. Per patient and modality: take the earliest study; within it, the eligible
series with the most images. A series is not eligible if its description matches the exclusion pattern or it has
fewer than `MIN_IMAGES` images. If the earliest study has no eligible series, the next study is used and this is recorded.
Each collection's **primary modality** is the one (CT or MR) with more patients; a tie goes to CT.

In [10]:
# 10. Step D: main series
desc = img.SeriesDescription.where(img.SeriesDescription.notna(), img.ProtocolName).fillna('')
img['excluded_pattern'] = desc.str.contains(EXCLUDE_SERIES_PATTERN, flags=re.I, regex=True)
img['excluded_ct_reformat'] = (img.Modality == 'CT') & desc.str.contains(CT_REFORMAT_PATTERN, flags=re.I, regex=True)
img['excluded_third_party'] = (img.ThirdPartyAnalysis.astype(str).str.strip().str.lower() == 'yes'
                               if 'ThirdPartyAnalysis' in img else False)
img['excluded_small'] = img.ImageCount.fillna(0) < MIN_IMAGES
img['eligible'] = ~(img.excluded_pattern | img.excluded_ct_reformat | img.excluded_third_party | img.excluded_small)

excl = (img.groupby('Collection')
           .agg(n_series_CT_MR=('SeriesInstanceUID', 'size'),
                n_excluded_pattern=('excluded_pattern', 'sum'),
                n_excluded_ct_reformat=('excluded_ct_reformat', 'sum'),
                n_excluded_third_party=('excluded_third_party', 'sum'),
                n_excluded_small=('excluded_small', 'sum'),
                n_eligible=('eligible', 'sum'))
           .reset_index().rename(columns={'Collection': 'collection'}))
excl.to_csv(OUT / 'series_exclusions.csv', index=False)

pts_by_mod = img.groupby(['Collection', 'Modality']).PatientID.nunique().unstack(fill_value=0)
primary_modality = {c: ('CT' if r.get('CT', 0) >= r.get('MR', 0) else 'MR') for c, r in pts_by_mod.iterrows()}

main_rows, no_main = [], []
for (col, mod, pid), g in img.groupby(['Collection', 'Modality', 'PatientID'], sort=True):
    order = (g.groupby('StudyInstanceUID').StudyDate_dt.min().reset_index()
              .sort_values(['StudyDate_dt', 'StudyInstanceUID'], na_position='last'))
    for rank, suid in enumerate(order.StudyInstanceUID, 1):
        e = g[(g.StudyInstanceUID == suid) & g.eligible]
        if len(e):
            b = e.sort_values(['ImageCount', 'SeriesNumber', 'SeriesInstanceUID'],
                              ascending=[False, True, True]).iloc[0]
            main_rows.append({'collection': col, 'PatientID': pid, 'modality': mod,
                              'is_primary_modality': mod == primary_modality[col],
                              'StudyInstanceUID': suid, 'StudyDate': b.StudyDate,
                              'study_rank_used': rank, 'n_studies_this_modality': len(order),
                              'n_series_this_modality': len(g),
                              'SeriesInstanceUID': b.SeriesInstanceUID, 'SeriesNumber': b.SeriesNumber,
                              'SeriesDescription': b.SeriesDescription, 'ProtocolName': b.ProtocolName,
                              'Manufacturer': b.Manufacturer, 'ManufacturerModelName': b.ManufacturerModelName,
                              'SoftwareVersions': b.SoftwareVersions, 'ImageCount': b.ImageCount})
            break
    else:
        no_main.append({'collection': col, 'PatientID': pid, 'modality': mod})

main = pd.DataFrame(main_rows)
main.to_csv(OUT / 'main_series.csv', index=False)
pd.DataFrame(no_main, columns=['collection', 'PatientID', 'modality']).to_csv(OUT / 'patients_without_main_series.csv', index=False)
print(f'Main series found for {len(main):,} patient-modality pairs; none found for {len(no_main):,}.')
print('Used a later study (earliest had no eligible series):', int((main.study_rank_used > 1).sum()))
print('Primary modality per collection:', primary_modality)

Main series found for 4,883 patient-modality pairs; none found for 17.
Used a later study (earliest had no eligible series): 52
Primary modality per collection: {'CPTAC-AML': 'CT', 'CPTAC-CCRCC': 'CT', 'CPTAC-CM': 'CT', 'CPTAC-LSCC': 'CT', 'CPTAC-LUAD': 'CT', 'CPTAC-PDA': 'CT', 'CPTAC-SAR': 'CT', 'CPTAC-STAD': 'CT', 'CPTAC-UCEC': 'CT', 'Duke-Breast-Cancer-MRI': 'MR', 'HCC-TACE-Seg': 'CT', 'LIDC-IDRI': 'CT', 'NSCLC Radiogenomics': 'CT', 'PROSTATEx': 'MR', 'TCGA-BLCA': 'CT', 'TCGA-BRCA': 'MR', 'TCGA-CESC': 'MR', 'TCGA-COAD': 'CT', 'TCGA-ESCA': 'CT', 'TCGA-KICH': 'CT', 'TCGA-KIRC': 'CT', 'TCGA-KIRP': 'CT', 'TCGA-LIHC': 'CT', 'TCGA-LUAD': 'CT', 'TCGA-LUSC': 'CT', 'TCGA-OV': 'CT', 'TCGA-PRAD': 'MR', 'TCGA-READ': 'CT', 'TCGA-SARC': 'CT', 'TCGA-STAD': 'CT', 'TCGA-THCA': 'CT', 'TCGA-UCEC': 'CT', 'UPENN-GBM': 'MR'}


## Step E · Labels from the GDC API (TCGA and CPTAC patients)

In [11]:
# 11. Step E: GDC clinical labels
GDC_FIELDS = ','.join(['submitter_id', 'project.project_id', 'disease_type', 'primary_site',
                       'diagnoses.primary_diagnosis', 'diagnoses.morphology', 'diagnoses.tumor_grade',
                       'diagnoses.ajcc_pathologic_stage', 'diagnoses.ajcc_clinical_stage',
                       'diagnoses.year_of_diagnosis', 'diagnoses.diagnosis_is_primary_disease'])
gdc_ids = sorted(series.loc[series.Collection.str.startswith(('TCGA-', 'CPTAC-')), 'PatientID'].dropna().unique())
hits = []
for i in range(0, len(gdc_ids), 300):
    batch = gdc_ids[i:i + 300]
    payload = {'filters': {'op': 'in', 'content': {'field': 'submitter_id', 'value': batch}},
               'fields': GDC_FIELDS, 'format': 'json', 'size': 3 * len(batch)}
    try:
        res = fetch_json(f'{GDC}/cases', payload=payload)['data']
    except Exception as e:
        FAILURES.append({'step': 'E', 'collection': f'GDC batch {i}', 'error': repr(e)})
        continue
    if res['pagination']['total'] > len(res['hits']):
        raise RuntimeError('GDC returned more cases than one page holds: raise "size".')
    hits += res['hits']

lab = []
for h in hits:
    diags = h.get('diagnoses') or []
    prim = [d for d in diags if d.get('diagnosis_is_primary_disease') is True]
    d = prim[0] if prim else (diags[0] if diags else {})
    lab.append({'PatientID': h['submitter_id'], 'gdc_project': h.get('project', {}).get('project_id'),
                'disease_type': h.get('disease_type'), 'primary_site': h.get('primary_site'),
                'diagnosis_choice': 'primary' if prim else ('first listed' if diags else 'none'),
                'n_primary_diagnoses': len(prim),
                'primary_diagnosis': d.get('primary_diagnosis'), 'morphology': d.get('morphology'),
                'tumor_grade': d.get('tumor_grade'), 'ajcc_pathologic_stage': d.get('ajcc_pathologic_stage'),
                'ajcc_clinical_stage': d.get('ajcc_clinical_stage'), 'year_of_diagnosis': d.get('year_of_diagnosis')})
gdc = pd.DataFrame(lab, columns=['PatientID', 'gdc_project', 'disease_type', 'primary_site', 'diagnosis_choice',
                                 'n_primary_diagnoses', 'primary_diagnosis', 'morphology', 'tumor_grade',
                                 'ajcc_pathologic_stage', 'ajcc_clinical_stage', 'year_of_diagnosis'])
gdc['n_gdc_records'] = gdc.groupby('PatientID').PatientID.transform('size')
gdc.to_csv(OUT / 'gdc_labels.csv', index=False)
print(f'GDC: {len(gdc_ids):,} TCGA/CPTAC patients asked, {gdc.PatientID.nunique():,} found, '
      f'{(gdc.n_gdc_records > 1).sum()} rows from patients with more than one GDC record.')

GDC: 1,566 TCGA/CPTAC patients asked, 1,479 found, 0 rows from patients with more than one GDC record.


## Step F · Which tasks have enough patients? (rule 7a.1)
Counts are patients with a main series in the collection's primary modality.
The binary groupings below are **proposals** for review; the final grouping is fixed before notebook 03.

In [12]:
# 12. Step F: label counts and task eligibility
def grade_group(g):
    if not isinstance(g, str):
        return None
    g = g.strip().lower()
    if g in ('g1', 'g2', 'low grade'):
        return 'low (G1-G2)'
    if g in ('g3', 'g4', 'high grade'):
        return 'high (G3-G4)'
    return None

def stage_group(s):
    m = re.match(r'\s*stage\s+([iv]+)', s.lower()) if isinstance(s, str) else None
    if not m:
        return None
    return 'early (I-II)' if m.group(1) in ('i', 'ii') else ('late (III-IV)' if m.group(1) in ('iii', 'iv') else None)

gdc1 = gdc.drop_duplicates('PatientID')   # patients with >1 GDC record are flagged in gdc_labels.csv
mp = main[main.is_primary_modality].merge(gdc1, on='PatientID', how='left')
mp['grade_group'] = mp.tumor_grade.map(grade_group)
mp['stage_group'] = mp.ajcc_pathologic_stage.map(stage_group)

raw = []
for f in ('tumor_grade', 'ajcc_pathologic_stage', 'ajcc_clinical_stage', 'primary_diagnosis',
          'grade_group', 'stage_group'):
    c = (mp[mp.collection.str.startswith(('TCGA-', 'CPTAC-'))]
         .groupby('collection')[f].value_counts(dropna=False).rename('n_patients').reset_index()
         .rename(columns={f: 'value'}))
    c.insert(1, 'field', f)
    raw.append(c)
raw = pd.concat(raw, ignore_index=True)
raw['value'] = raw['value'].fillna('(missing)')
raw.to_csv(OUT / 'label_counts_raw.csv', index=False)

elig = []
for col, g in mp[mp.collection.str.startswith(('TCGA-', 'CPTAC-'))].groupby('collection'):
    for task, f in (('grade low vs high', 'grade_group'), ('stage early vs late', 'stage_group')):
        vc = g[f].value_counts()
        elig.append({'task_type': 'within collection (6a)', 'task': task, 'collections': col,
                     'modality': primary_modality[col],
                     'class_counts': '; '.join(f'{k}: {v}' for k, v in vc.items()) or 'none',
                     'n_classes': len(vc), 'min_class_n': int(vc.min()) if len(vc) else 0})
# Three-class tasks are also counted pair by pair, to show which pairs could stand alone
from itertools import combinations
defined = dict(COLLECTION_DEFINED_TASKS)
for task, cols in COLLECTION_DEFINED_TASKS.items():
    if len(cols) > 2:
        for a, b in combinations(cols, 2):
            defined[f'{task}: {a} vs {b}'] = [a, b]
for task, cols in defined.items():
    for mod in IMAGING_MODALITIES:
        counts = {c: int(((main.collection == c) & (main.modality == mod)).sum()) for c in cols}
        absent = [c for c in cols if c not in collections]
        elig.append({'task_type': 'collection-defined (6b)', 'task': task, 'collections': ', '.join(cols),
                     'modality': mod,
                     'class_counts': '; '.join(f'{c}: {n}' + (' (not on TCIA API)' if c in absent else '')
                                               for c, n in counts.items()),
                     'n_classes': sum(n > 0 for n in counts.values()), 'min_class_n': min(counts.values())})
elig = pd.DataFrame(elig)
elig['eligible'] = (elig.n_classes >= 2) & (elig.min_class_n >= MIN_PER_CLASS)
elig.to_csv(OUT / 'task_eligibility.csv', index=False)
print(f'Tasks meeting rule 7a.1 (>= {MIN_PER_CLASS} patients per class): {int(elig.eligible.sum())} of {len(elig)}')
show(elig.sort_values(['task_type', 'eligible', 'min_class_n'], ascending=[True, False, False]), 80)

Tasks meeting rule 7a.1 (>= 20 patients per class): 14 of 66


,task_type,task,collections,modality,class_counts,n_classes,min_class_n,eligible
58,collection-defined (6b),lung_histology,"TCGA-LUAD, TCGA-LUSC",CT,TCGA-LUAD: 67; TCGA-LUSC: 37,2,37,True
60,collection-defined (6b),renal_subtype: TCGA-KIRC vs TCGA-KIRP,"TCGA-KIRC, TCGA-KIRP",CT,TCGA-KIRC: 236; TCGA-KIRP: 23,2,23,True
61,collection-defined (6b),renal_subtype: TCGA-KIRC vs TCGA-KIRP,"TCGA-KIRC, TCGA-KIRP",MR,TCGA-KIRC: 62; TCGA-KIRP: 17,2,17,False
56,collection-defined (6b),renal_subtype,"TCGA-KIRC, TCGA-KIRP, TCGA-KICH",CT,TCGA-KIRC: 236; TCGA-KIRP: 23; TCGA-KICH: 12,3,12,False
62,collection-defined (6b),renal_subtype: TCGA-KIRC vs TCGA-KICH,"TCGA-KIRC, TCGA-KICH",CT,TCGA-KIRC: 236; TCGA-KICH: 12,2,12,False
...,...,...,...,...,...,...,...,...
44,within collection (6a),grade low vs high,TCGA-READ,CT,none,0,0,False
46,within collection (6a),grade low vs high,TCGA-SARC,CT,none,0,0,False
47,within collection (6a),stage early vs late,TCGA-SARC,CT,none,0,0,False
50,within collection (6a),grade low vs high,TCGA-THCA,CT,none,0,0,False


In [13]:
# 13. Patients that appear in more than one candidate collection (matters for collection-defined tasks)
pc = series.groupby('PatientID').Collection.agg(lambda s: sorted(set(s)))
overlap = pc[pc.map(len) > 1].map('; '.join).rename('collections').reset_index()
overlap.to_csv(OUT / 'collection_overlap.csv', index=False)
print('Patients in more than one candidate collection:', len(overlap))

Patients in more than one candidate collection: 0


## Step G · Do the dates look real? (rule 7a.5)
For TCGA and CPTAC, the study year is compared with the GDC year of diagnosis. Real dates should mostly agree within
two years. The verdict is only a **suggestion** with fixed thresholds (≥ 80% within 2 years → looks real;
≤ 20% → looks shifted); the final decision per collection is ours, logged in `00_LOG.md`.

In [14]:
# 14. Step G: date plausibility
d = main[main.is_primary_modality].merge(gdc1[['PatientID', 'year_of_diagnosis']], on='PatientID', how='left')
d['study_year'] = pd.to_datetime(d.StudyDate, errors='coerce').dt.year
d['diff_years'] = d.study_year - pd.to_numeric(d.year_of_diagnosis, errors='coerce')
rel = pd.to_datetime(series.groupby('Collection').DateReleased.max(), errors='coerce')
dates = []
for col in candidates:
    g = d[d.collection == col]
    both = g.dropna(subset=['diff_years'])
    s = series[series.Collection == col]
    sd = pd.to_datetime(s.StudyDate, errors='coerce')
    n_both = len(both)
    frac = float((both.diff_years.abs() <= 2).mean()) if n_both else np.nan
    if n_both < 10:
        verdict = 'no reference (fewer than 10 patients with a diagnosis year)'
    elif frac >= 0.8:
        verdict = 'looks real'
    elif frac <= 0.2:
        verdict = 'looks shifted'
    else:
        verdict = 'unclear'
    dates.append({'collection': col, 'min_study_year': sd.dt.year.min(), 'max_study_year': sd.dt.year.max(),
                  'fraction_study_dates_missing': round(float(sd.isna().mean()), 3),
                  'n_series_dated_after_release': int((sd > rel.get(col)).sum()) if pd.notna(rel.get(col)) else np.nan,
                  'n_patients_with_diagnosis_year': n_both,
                  'median_study_minus_diagnosis_year': both.diff_years.median() if n_both else np.nan,
                  'fraction_within_2_years': round(frac, 3) if n_both else np.nan,
                  'suggested_verdict': verdict})
dates = pd.DataFrame(dates)
dates.to_csv(OUT / 'dates_check.csv', index=False)
show(dates, 40)

,collection,min_study_year,max_study_year,fraction_study_dates_missing,n_series_dated_after_release,n_patients_with_diagnosis_year,median_study_minus_diagnosis_year,fraction_within_2_years,suggested_verdict
0,CPTAC-AML,2012,2013,0.0,0,5,-7.0,0.000,no reference (fewer than 10 patients with a diagnosis year)
1,CPTAC-CCRCC,1999,2012,0.0,0,64,-8.0,0.000,looks shifted
2,CPTAC-CM,2000,2012,0.0,0,14,-8.0,0.071,looks shifted
3,CPTAC-LSCC,1999,2011,0.0,0,22,-15.0,0.000,looks shifted
4,CPTAC-LUAD,1999,2013,0.0,0,36,-8.0,0.000,looks shifted
5,CPTAC-PDA,1999,2012,0.0,0,69,-14.0,0.000,looks shifted
6,CPTAC-SAR,1999,2012,0.0,0,0,NaN,NaN,no reference (fewer than 10 patients with a diagnosis year)
7,CPTAC-STAD,2005,2017,0.0,0,19,-8.0,0.000,looks shifted
8,CPTAC-UCEC,1998,2020,0.0,0,69,-16.0,0.043,looks shifted
9,TCGA-BLCA,1986,2007,0.0,0,107,-10.0,0.009,looks shifted


## Step H · Fields that could leak the diagnosis, and label sources for non-TCGA collections

In [15]:
# 15. Step H: leak check on study-level free text
leak = []
for col, g in studies.groupby('Collection'):
    for f in ('AdmittingDiagnosesDescription',):
        if f in g:
            v = g[f].dropna()
            leak.append({'collection': col, 'field': f, 'fraction_filled': round(float(g[f].notna().mean()), 3),
                         'n_distinct': v.nunique(), 'examples': ' | '.join(map(str, v.value_counts().index[:5]))})
leak = pd.DataFrame(leak, columns=['collection', 'field', 'fraction_filled', 'n_distinct', 'examples'])
leak.to_csv(OUT / 'leak_fields_check.csv', index=False)
print('Collections where AdmittingDiagnosesDescription is filled:', int((leak.fraction_filled > 0).sum()))

Collections where AdmittingDiagnosesDescription is filled: 15


In [16]:
# 16. Label sources still to locate (non-TCGA candidates)
todo = []
for col, label in CANDIDATES_NON_TCGA.items():
    s = series[series.Collection == col]
    m = main[(main.collection == col) & main.is_primary_modality]
    todo.append({'collection': col, 'on_tcia_api': col in collections,
                 'doi': '; '.join(sorted({u.lower() for u in s.CollectionURI.dropna()})) if 'CollectionURI' in s else '',
                 'primary_modality': primary_modality.get(col), 'n_patients_with_main_series': len(m),
                 'label_planned': label,
                 'label_file_status': 'to locate on the DOI page; save under labels/' + re.sub(r'\W+', '_', col)})
todo = pd.DataFrame(todo)
todo.to_csv(OUT / 'label_sources_todo.csv', index=False)
readme = LABELS / 'README.txt'
if not readme.exists():
    readme.write_text('Clinical label files for non-TCGA collections, one subfolder per collection.\n'
                      'Each file is downloaded by hand from the collection DOI page, unchanged.\n'
                      'Record the file name, source URL and download date in 00_LOG.md.\n')
show(todo)

,collection,on_tcia_api,doi,primary_modality,n_patients_with_main_series,label_planned,label_file_status
0,LIDC-IDRI,True,http://doi.org/10.7937/k9/tcia.2015.1buvfjr7; https://do...,CT,1010,nodule malignancy rating (radiologists),to locate on the DOI page; save under labels/LIDC_IDRI
1,Duke-Breast-Cancer-MRI,True,https://doi.org/10.7937/tcia.e3sv-re93,MR,922,molecular subtype,to locate on the DOI page; save under labels/Duke_Breast...
2,NSCLC Radiogenomics,True,https://doi.org/10.7937/k9/tcia.2017.7hs46erv,CT,203,histology; mutations,to locate on the DOI page; save under labels/NSCLC_Radio...
3,PROSTATEx,True,https//doi.org/10.7937/tcia.nbb4-4655; https://doi.org/1...,MR,346,clinical significance,to locate on the DOI page; save under labels/PROSTATEx
4,UPENN-GBM,True,https://doi.org/10.7937/tcia.709x-dn49,MR,630,molecular markers,to locate on the DOI page; save under labels/UPENN_GBM
5,HCC-TACE-Seg,True,https://doi.org/10.7937/tcia.5fna-0924,CT,105,to define from the clinical file,to locate on the DOI page; save under labels/HCC_TACE_Seg


## Step I · Are the missing collections in the Imaging Data Commons (IDC)?
Optional check. IDC mirrors TCIA's public collections and offers series metadata without an account.
If this step fails, nothing else is affected.

In [17]:
# 17. Step I: IDC availability of the collections missing from the TCIA API
idc_rows = []
try:
    import subprocess
    if IN_COLAB:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'idc-index'], check=True)
    from idc_index import IDCClient
    idx = IDCClient().index
    (OUT / 'idc_index_columns.txt').write_text('\n'.join(map(str, idx.columns)))
    norm = lambda s: re.sub(r'[^a-z0-9]', '_', s.lower())
    for col in sorted(planned):
        g = idx[idx.collection_id.map(norm) == norm(col)]
        idc_rows.append({'collection': col, 'on_tcia_api': col in collections, 'in_idc': len(g) > 0,
                         'idc_patients': g.PatientID.nunique() if len(g) else 0,
                         'idc_series': len(g),
                         'idc_modalities': ';'.join(sorted(g.Modality.dropna().unique())) if len(g) else '',
                         'idc_has_CT_or_MR': bool(g.Modality.isin(IMAGING_MODALITIES).any())})
    print('idc-index version:', __import__('idc_index').__version__)
except Exception as e:
    FAILURES.append({'step': 'I', 'collection': 'IDC check', 'error': repr(e)})
    print('IDC check skipped:', repr(e))
idc = pd.DataFrame(idc_rows, columns=['collection', 'on_tcia_api', 'in_idc', 'idc_patients', 'idc_series',
                                      'idc_modalities', 'idc_has_CT_or_MR'])
idc.to_csv(OUT / 'idc_availability.csv', index=False)
show(idc)

idc-index version: 0.13.4


,collection,on_tcia_api,in_idc,idc_patients,idc_series,idc_modalities,idc_has_CT_or_MR
0,Duke-Breast-Cancer-MRI,True,True,922,6058,MR;SEG,True
1,HCC-TACE-Seg,True,True,105,1020,CT;SEG,True
2,LIDC-IDRI,True,True,1010,15116,CR;CT;DX;SEG;SR,True
3,NSCLC Radiogenomics,True,True,211,1722,CT;PT;SEG,True
4,PROSTATEx,True,True,346,19177,MR;SEG;SR,True
5,TCGA-GBM,False,True,607,4286,ANN;OT;SEG;SM,False
6,TCGA-KICH,True,True,113,468,CT;MR;SEG;SM,True
7,TCGA-KIRC,True,True,537,6123,CR;CT;MR;SEG;SM,True
8,TCGA-KIRP,True,True,291,1194,CT;MR;PT;SEG;SM,True
9,TCGA-LGG,False,True,516,1572,SM,False


## Step J · Run record and summary

In [18]:
# 18. Step J: manifest (what was run, and a checksum for every output file)
def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

files = []
for p in sorted(OUT.iterdir()):
    if p.name == 'run_manifest.json':
        continue
    n_rows = None
    if p.suffix == '.csv' or p.name.endswith('.csv.gz'):
        n_rows = len(pd.read_csv(p, low_memory=False))
    files.append({'file': p.name, 'bytes': p.stat().st_size, 'rows': n_rows, 'md5': md5(p)})

manifest = {
    'notebook': '01_tcia_inventory.ipynb', 'run_started_utc': RUN_STARTED,
    'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'in_colab': IN_COLAB, 'test_collections': TEST_COLLECTIONS,
    'apis': {'tcia': TCIA, 'gdc': GDC}, 'api_calls': CALLS, 'failures': FAILURES,
    'settings': {'MIN_PER_CLASS': MIN_PER_CLASS, 'MIN_IMAGES': MIN_IMAGES,
                 'IMAGING_MODALITIES': IMAGING_MODALITIES, 'EXCLUDE_SERIES_PATTERN': EXCLUDE_SERIES_PATTERN,
                 'CANDIDATES_NON_TCGA': CANDIDATES_NON_TCGA, 'COLLECTION_DEFINED_TASKS': COLLECTION_DEFINED_TASKS},
    'candidates': candidates, 'primary_modality': primary_modality,
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__,
                 'numpy': np.__version__, 'requests': requests.__version__},
    'outputs': files,
}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(f"API calls: {CALLS['network']} over the network, {CALLS['cache']} from cache.")
print('Failures:', FAILURES or 'none')
show(pd.DataFrame(files), 40)

API calls: 541 over the network, 0 from cache.
Failures: none


,file,bytes,rows,md5
0,candidates_missing.csv,67,2.0,f808bb7733f8a4d1c5a96ff62c41e158
1,collection_overlap.csv,22,0.0,d339e586351a7d09a5a0aa89c4ecd4cf
2,collections_inventory.csv,16280,156.0,78ed7931f1f0dd1c17326a794192d43e
3,dates_check.csv,2423,33.0,ceeec69eb2d74dfd80cf40efa1d14df8
4,field_roles.csv,2381,38.0,4850a8efb6c5b9f196c224317107b88e
5,gdc_labels.csv,196177,1479.0,38c8ddf5ba26dccdc45ac1eabbb199f4
6,idc_availability.csv,717,13.0,b0b2e23eca82e86fa50ea488f88fa462
7,idc_index_columns.txt,494,NaN,170bf0286289d93c17ca948c46bb9de9
8,label_counts_raw.csv,17877,422.0,d7d3a2056f5865dc0dad07e8a7d7ff07
9,label_sources_todo.csv,1197,6.0,0d144c1d74abb7c1a372524b15c43d6d


In [19]:
# 19. Plain-English summary, read back from the saved files
inv_s = pd.read_csv(OUT / 'collections_inventory.csv')
el_s = pd.read_csv(OUT / 'task_eligibility.csv')
ms_s = pd.read_csv(OUT / 'main_series.csv')
print(f"- TCIA public API lists {len(inv_s)} collections; {int(inv_s.human_ct_or_mr.sum())} are human collections with CT or MR.")
print(f"- Planned collections missing from the API: {', '.join(pd.read_csv(OUT / 'candidates_missing.csv').collection) or 'none'}.")
print(f"- Candidate collections pulled: {len(candidates)}; patients with a main series in the primary modality: "
      f"{int(ms_s.is_primary_modality.sum()):,}.")
for t, g in el_s.groupby('task_type'):
    print(f"- {t}: {int(g.eligible.sum())} of {len(g)} task rows have at least {MIN_PER_CLASS} patients per class.")
print('\nNext: review results/01_inventory/ and its run log from Drive.')

- TCIA public API lists 156 collections; 114 are human collections with CT or MR.
- Planned collections missing from the API: TCGA-GBM, TCGA-LGG.
- Candidate collections pulled: 33; patients with a main series in the primary modality: 4,644.
- collection-defined (6b): 2 of 12 task rows have at least 20 patients per class.
- within collection (6a): 12 of 54 task rows have at least 20 patients per class.

Next: review results/01_inventory/ and its run log from Drive.
